# Per-level privacy report

Measures the files written by `anonymise.run` for a level: how many stays sit in small cells on the released static fields, and how well the hospital can be guessed from the released files. The classifier is gradient boosting (100 iterations, untuned) with 5-fold cross-validation grouped by patient; the baseline is always guessing the largest site. Accuracy is a lower bound on what a better attacker could do. Site labels come from the private key map and are used only to score the guesses. Only counts, shares and accuracies leave the SDE, and figures need the output check.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))

import level_report as LR

In [ ]:
derived_dir = open_picu_dir / "derived"
reports_dir = open_picu_dir / "reports"
out_root = derived_dir / "anonymised"
static = pd.read_parquet(derived_dir / "static" / "pooled_static.parquet")
level = "tier2"

## 1. Small cells on the released static fields

Stays in cells below k on age band, sex, ethnicity, diagnosis group, outcome and weight band, all sites together. The attributes show the fields used and the smallest cell. The check in the run allows 0.1% below k = 10.

In [ ]:
cs = LR.cell_summary(out_root, level)
print(cs.attrs)
cs

## 2. Can the site be guessed?

Four variants: the released static fields; which concepts a stay has any rows of; both together; and both plus which drugs a stay received. Accuracy against the baseline, the gain in percentage points, and the share of each site's stays guessed correctly. This runs five folds on each variant and takes a few minutes.

In [ ]:
res = LR.site_report(out_root, level, static)
LR.summary(res)

## 3. Figure

Accuracy per variant against the baseline, and recall by site. Check it with the output checker before it leaves the SDE.

In [ ]:
LR.plot(res, title="Tier 2: guessing the site from the released files", path=reports_dir / "tier2_site_inference.png");

## 4. Comparing levels

Site inference on each level's released files, one table. The classifier takes several minutes per level, so each result is saved on first use (in `_private` under the output folder, accuracies and recalls only) and loaded afterwards. A level is recomputed automatically if it has been run again since. Add a level to the list once it has been run; pass `refresh=True` to force a recompute.

In [ ]:
levels = ["tier2", "tier2_no_ethnicity", "tier2_shared", "tier2_with_caboodle"]
tab, results = LR.compare_levels(out_root, levels, static)
tab

## 5. Which concepts give the site away?
1. Coverage: which concepts exist at only one or two sites.
2. Single-concept accuracy: how much each concept's presence alone reveals.
3. Removal curve: how far accuracy falls as the top concepts are removed.
4. Shift: do shared concepts have different values at different sites (possible processing problem)?

### Coverage and single-concept accuracy

`concept_coverage_released` gives, for each concept, the share of each site's stays with at least one row, and how many sites record it (at least 1% of stays). The first table lists only concepts at fewer than three sites.

`single_concept_accuracy` guesses the site from one concept's presence alone ("has rows" or "has none"). It is exact, with no model. `gain_pts` is the accuracy above the baseline of always guessing the largest site, in percentage points. The top rows are the concepts that give the site away, and they are candidates to drop or merge.

In [ ]:
assert hasattr(LR, "concept_shift"), "old level_report.py"

name = "tier2"
cov = LR.concept_coverage_released(out_root, name)
one = LR.single_concept_accuracy(out_root, name)
print("baseline:", round(one.attrs["baseline"], 3))
display(cov[cov["n_sites"] < 3])
display(one.head(25))

### Removal curve

The classifier is the one used in the site report (gradient boosting, grouped cross-validation by patient). It is run on which concepts a stay has, after the top 0, 3, 6, ... most revealing concepts (from the table above) are removed. `last_removed` is the last concept removed at each step. This shows how many concepts have to go before the site is no longer certain. Any accuracy that stays high after the one-site and two-site concepts are gone comes from recording-practice differences at shared concepts.

In [ ]:
curve = LR.removal_curve(out_root, name, static, steps=(0, 3, 6, 10, 15, 20, 30))
display(curve)

### Value shift on shared concepts

For every numeric concept recorded at two or more sites (at least 10 stays each), `concept_shift` takes the per-stay median value and compares the sites.

- `shift`: the largest difference between two sites' medians, in units of the typical within-site spread (IQR / 1.349). Around 0 means the sites look alike; large means they differ.
- `ks`: the largest Kolmogorov-Smirnov distance between two sites (0 identical, 1 fully separated).
- `ratio` and `scale_hint`: the ratio of the two medians, and the name of a common unit factor if it is within 5% of one (x10, x100, x1000, x60, kPa/mmHg, °C/°F).

A large shift with a scale hint points to a unit or processing problem, which is worth fixing in harmonisation. A large shift without one may be a real difference in case mix or practice, so check it with a clinician before treating it as an error. The second table shows only the concepts that carry a scale hint.

In [ ]:
shift = LR.concept_shift(out_root, name)
cols = ["concept", "n_sites", "shift", "ks", "pair", "ratio", "scale_hint"]
display(shift[cols].head(30))
display(shift[shift["scale_hint"] != ""][cols])

### What gives the site away in `tier2_shared`?

The 58 concepts still released at `tier2_shared` all exist at three sites. Their presence still gives the site away at 89.5%, so the signal is how often each site records them. `spread` is the largest minus the smallest site share of stays with the concept. A large spread means that site records it much more or less often. The single-concept table shows how much each one reveals alone, and the removal curve shows how quickly accuracy falls as they go.

In [ ]:
name = "tier2_shared"
cov = LR.concept_coverage_released(out_root, name)
one = LR.single_concept_accuracy(out_root, name)
display(cov.sort_values("spread", ascending=False).head(20))
display(one.head(20))
LR.removal_curve(out_root, name, static, steps=(0, 5, 10, 15, 20, 30))

### Site inference with shared concepts only

Same report as for `tier2`, run on `tier2_shared` (every concept recorded at fewer than three sites dropped). The table puts it next to `tier2_no_ethnicity`, which has the same settings except for the dropped concepts. The drop in accuracy is what removing those 20 concepts buys. What remains is site signal in how often each site records the shared concepts. Accuracy is cross-validated and grouped by patient, against the baseline of always guessing the largest site.

In [ ]:
res_shared = LR.site_report(out_root, "tier2_shared", static)
tab = pd.concat({n: LR.summary(r) for n, r in {"tier2_no_ethnicity": res_ne, "tier2_shared": res_shared}.items()}, names=["level"]).reset_index(level=0)
display(tab)
display(LR.cell_summary(out_root, "tier2_shared"))